In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import  ColumnTransformer
from sklearn.linear_model import  LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import *


Chargement des données

In [17]:
df = pd.read_csv('dataset.csv')

In [19]:
df.drop_duplicates(inplace=True)

Decoupage 80/20

In [10]:
x  = df[df.columns.difference(['Exam_Score'])]
y = df[['Exam_Score']]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

Definir le Transformer pour les nombres numeriques

In [21]:
from sklearn.base import BaseEstimator, TransformerMixin

class IGRClipper(BaseEstimator, TransformerMixin):

      def fit(self, X, y=None):
            #claculer du premier quartier
            self.Q1 = np.percentile(X, 25, axis=0)

            #calcul du troisieme quartie
            self.Q3 = np.percentile(X, 75, axis=0)

            #calcul de l'IQR
            self.IQR = self.Q3 - self.Q1

            #bornes inferieure et superieure
            self.lower_bound_ = self.Q1 - 1.5 * self.IQR
            self.upper_bound_ = self.Q3 + 1.5 * self.IQR

            return self

      def transform(self, X):
            X = np.asarray(X)

            #limiter les valeurs aux bornes IQR
            X_clipped = np.clip(
                  X,
                  self.lower_bound_,
                  self.upper_bound_
            )
            # _ Cette valeur a été apprise pendant fit().
            return X_clipped


In [23]:
numeric_transformer = Pipeline(steps=[
      ('imputer', SimpleImputer(strategy='median')),
      ('iqr', IGRClipper()),
      ('scaler', StandardScaler())
])

In [24]:
categorical_transformer1 = Pipeline(steps=[
      ('imputer', SimpleImputer(strategy='most_frequent')),
      ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [25]:
categorical_transformer2 = Pipeline(steps=[
      ('imputer', SimpleImputer(strategy='most_frequent')),
      ('OrdinalEncoder', OrdinalEncoder())
])

In [ ]:
features_preprocessor = ColumnTransformer(
      transformers=[
            ('numeric', numeric_transformer, [])
      ]
)